# Session 3: Cyber Laws, Data Protection and Digital Evidence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/05-cyber-security-frameworks/code/session-03-cyber-laws-and-digital-evidence.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-03_custody_ledger.py`

In [1]:
"""Session 3 - A hash-chained chain-of-custody ledger.

Courts accept digital evidence only if the prosecution can show who held it, when,
why, and that it never changed. A paper custody form can be rewritten quietly; this
ledger cannot. Each entry stores the SHA-256 hash of the previous entry, so editing
any old line breaks every link after it (the same idea a blockchain uses).

All people, cases and evidence below are fictional. The script uses fixed times so
the output is the same on every run.
Run:  python session-03_custody_ledger.py
"""
import hashlib
import json

GENESIS = "0" * 64


def sha256(data):
    return hashlib.sha256(data).hexdigest()


def entry_hash(entry):
    """Hash every field except the entry's own hash, in a fixed key order."""
    body = {k: v for k, v in entry.items() if k != "hash"}
    return sha256(json.dumps(body, sort_keys=True).encode())


class CustodyLedger:
    def __init__(self, case_id):
        self.case_id = case_id
        self.entries = []

    def add(self, time, item, action, from_person, to_person, purpose, item_sha256):
        prev = self.entries[-1]["hash"] if self.entries else GENESIS
        entry = {
            "seq": len(self.entries) + 1, "case": self.case_id, "time": time,
            "item": item, "action": action, "from": from_person, "to": to_person,
            "purpose": purpose, "item_sha256": item_sha256, "prev": prev,
        }
        entry["hash"] = entry_hash(entry)
        self.entries.append(entry)
        return entry

    def verify(self, current_item_hashes):
        """Return a list of problems; an empty list means the ledger is intact."""
        problems, prev = [], GENESIS
        first_hash = {}
        for e in self.entries:
            if e["prev"] != prev:
                problems.append(f"entry {e['seq']}: link to previous entry is broken")
            if entry_hash(e) != e["hash"]:
                problems.append(f"entry {e['seq']}: contents were edited after signing")
            first_hash.setdefault(e["item"], e["item_sha256"])
            if e["item_sha256"] != first_hash[e["item"]]:
                problems.append(f"entry {e['seq']}: {e['item']} hash differs from acquisition")
            prev = e["hash"]
        for item, h in current_item_hashes.items():
            if h != first_hash.get(item):
                problems.append(f"{item}: evidence today does not match the acquisition hash")
        return problems


# --- fictional evidence (bytes stand in for real disk/phone images) ---
laptop_img = b"EXHIBIT HZ-01 laptop image: invoices.xlsx, mail.pst, browser history"
phone_img = b"EXHIBIT HZ-02 phone extraction: chats, call log, photos"
H1, H2 = sha256(laptop_img), sha256(phone_img)

ledger = CustodyLedger("KHI-CC-2026-0147")
ledger.add("2026-10-20 09:40 PKT", "HZ-01 laptop", "SEIZED", "suspect desk, Office 3", "Insp. Rabia", "Search under warrant", H1)
ledger.add("2026-10-20 09:55 PKT", "HZ-02 phone", "SEIZED", "suspect pocket", "Insp. Rabia", "Search under warrant", H2)
ledger.add("2026-10-20 13:10 PKT", "HZ-01 laptop", "TRANSFER", "Insp. Rabia", "Evidence room (Clerk Danish)", "Secure storage, bag seal 55871", H1)
ledger.add("2026-10-21 10:05 PKT", "HZ-01 laptop", "TRANSFER", "Clerk Danish", "Analyst Omer (forensic lab)", "Imaging with write blocker", H1)
ledger.add("2026-10-21 16:30 PKT", "HZ-01 laptop", "RETURN", "Analyst Omer", "Clerk Danish", "Imaging complete, seal 55902", H1)

print(f"Chain of custody for case {ledger.case_id}\n")
print(f"{'#':>2}  {'time':<20} {'item':<13} {'action':<9} {'from -> to':<52} hash")
for e in ledger.entries:
    who = f"{e['from']} -> {e['to']}"
    print(f"{e['seq']:>2}  {e['time']:<20} {e['item']:<13} {e['action']:<9} {who[:52]:<52} {e['hash'][:12]}..")
print(f"\nItem hashes at acquisition: HZ-01 {H1[:16]}..  HZ-02 {H2[:16]}..")

today = {"HZ-01 laptop": sha256(laptop_img), "HZ-02 phone": sha256(phone_img)}
problems = ledger.verify(today)
print("\n[1] Verify untouched ledger:", "INTACT" if not problems else problems)

# --- Scenario 2: someone quietly rewrites who held the laptop on 21 Oct ---
ledger.entries[3]["to"] = "Unknown contractor"
print("\n[2] Entry 4 edited to hide an unrecorded hand-over:")
for p in ledger.verify(today):
    print("   !", p)
ledger.entries[3]["to"] = "Analyst Omer (forensic lab)"      # undo for the next test

# --- Scenario 3: the forger also recomputes entry 4's own hash ---
ledger.entries[3]["to"] = "Unknown contractor"
ledger.entries[3]["hash"] = entry_hash(ledger.entries[3])
print("\n[3] Entry 4 edited AND its own hash recomputed:")
for p in ledger.verify(today):
    print("   !", p)
print("   -> the next entry still points at the old hash, so the chain exposes it.")
ledger.entries[3]["to"] = "Analyst Omer (forensic lab)"
ledger.entries[3]["hash"] = entry_hash(ledger.entries[3])

# --- Scenario 4: the evidence file itself changes in storage ---
tampered = laptop_img.replace(b"invoices", b"invoicez")
print("\n[4] One byte of the laptop image changed in storage:")
for p in ledger.verify({"HZ-01 laptop": sha256(tampered), "HZ-02 phone": H2}):
    print("   !", p)
print("\nTip: print the last entry hash on the paper form and in the case file;")
print("anyone can later recompute the chain and compare it with that number.")
print("Last entry hash:", ledger.entries[-1]["hash"])

Chain of custody for case KHI-CC-2026-0147

 #  time                 item          action    from -> to                                           hash
 1  2026-10-20 09:40 PKT HZ-01 laptop  SEIZED    suspect desk, Office 3 -> Insp. Rabia                bbb93f39328a..
 2  2026-10-20 09:55 PKT HZ-02 phone   SEIZED    suspect pocket -> Insp. Rabia                        2fcf194f72a4..
 3  2026-10-20 13:10 PKT HZ-01 laptop  TRANSFER  Insp. Rabia -> Evidence room (Clerk Danish)          f94fbe297094..
 4  2026-10-21 10:05 PKT HZ-01 laptop  TRANSFER  Clerk Danish -> Analyst Omer (forensic lab)          4fa7f6ac0c4f..
 5  2026-10-21 16:30 PKT HZ-01 laptop  RETURN    Analyst Omer -> Clerk Danish                         1bf550628951..

Item hashes at acquisition: HZ-01 dfed868a09e81fa6..  HZ-02 cdd47f69d1a8ae1a..

[1] Verify untouched ledger: INTACT

[2] Entry 4 edited to hide an unrecorded hand-over:
   ! entry 4: contents were edited after signing

[3] Entry 4 edited AND its own hash recomput

## `session-03_volatility_planner.py`

In [2]:
"""Session 3 - Plan evidence collection in the order of volatility.

Some evidence disappears in seconds (CPU state, RAM, network connections); some lasts
for years (backups, printouts). RFC 3227 and NIST SP 800-86 say: collect the most
volatile evidence first. This planner takes the artefacts on a fictional live machine,
sorts them by volatility, skips what is impossible in the device's state, warns about
traps (full-disk encryption, cloud sessions), and checks the plan fits the time on site.

Run:  python session-03_volatility_planner.py
"""

# rank: 1 = disappears fastest.  minutes = rough collection time.  needs_power: lost at shutdown
ARTEFACTS = [
    # name,                                   rank, minutes, needs_power
    ("Archival media (offsite backups)",          8, 0,  False),
    ("Disk image (with write blocker)",           6, 90, False),
    ("RAM (full memory capture)",                 2, 12, True),
    ("Network connections and ARP/route cache",   3, 2,  True),
    ("Running processes and open files",          4, 3,  True),
    ("Logged-in users and cloud sessions",        4, 2,  True),
    ("Temporary files and swap/pagefile",         5, 10, False),
    ("CPU registers and cache",                   1, 0,  True),
    ("Remote logs (firewall, SIEM, email)",       7, 20, False),
    ("Screen photo (what is visible now)",        1, 1,  True),
]

TOOLS = {
    "RAM": "memory imager (e.g. WinPmem/LiME) to external drive",
    "Network": "netstat/ss + arp -a output saved and hashed",
    "Running": "process list with command lines (e.g. tasklist /v, ps aux)",
    "Logged-in": "query user / who; note cloud tabs before they time out",
    "Disk": "imager + hardware write blocker, MD5+SHA-256",
    "Temporary": "collected inside the disk image; note pagefile/swap",
    "Remote": "preservation request to the log owner, export + hash",
    "Archival": "request through the organisation's backup custodian",
    "Screen": "photograph the screen with time visible",
    "CPU": "not practical to capture; noted for completeness",
}


def plan(device_on, disk_encrypted, minutes_on_site):
    print(f"\n=== Device {'ON' if device_on else 'OFF'} | full-disk encryption "
          f"{'YES' if disk_encrypted else 'no'} | {minutes_on_site} min on site ===")
    steps, used, skipped = [], 0, []
    for name, rank, mins, needs_power in sorted(ARTEFACTS, key=lambda a: (a[1], a[2])):
        if needs_power and not device_on:
            skipped.append(f"{name} (lost when the device powered off)")
            continue
        if mins == 0:
            skipped.append(f"{name} (not practical or done off site)")
            continue
        steps.append((rank, name, mins))
    for i, (rank, name, mins) in enumerate(steps, 1):
        used += mins
        flag = "" if used <= minutes_on_site else "  <- over time budget: hand to lab / preservation request"
        tool = TOOLS[name.split()[0]]
        print(f" {i:>2}. [vol {rank}] {name:<42} {mins:>3} min (t={used:>3})  {tool}{flag}")
    for s in skipped:
        print("     skip:", s)
    if device_on and disk_encrypted:
        print("  ! Encrypted disk is UNLOCKED now: capture RAM (keys) and consider a live logical")
        print("    copy of key folders BEFORE any shutdown, or the data may become unreadable.")
    if device_on:
        print("  ! Isolate from the network (unplug cable / airplane mode) rather than power off,")
        print("    unless destructive malware is actively wiping data.")
    else:
        print("  ! Device is off: do NOT switch it on. Bag, seal, label and image in the lab.")
    print("  ! Record every command you type, with the time, in your notes and custody form.")


# Case A: suspect laptop found running and unlocked, encrypted disk, 60 minutes on site
plan(device_on=True, disk_encrypted=True, minutes_on_site=60)
# Case B: same model found switched off
plan(device_on=False, disk_encrypted=True, minutes_on_site=60)


=== Device ON | full-disk encryption YES | 60 min on site ===
  1. [vol 1] Screen photo (what is visible now)           1 min (t=  1)  photograph the screen with time visible
  2. [vol 2] RAM (full memory capture)                   12 min (t= 13)  memory imager (e.g. WinPmem/LiME) to external drive
  3. [vol 3] Network connections and ARP/route cache      2 min (t= 15)  netstat/ss + arp -a output saved and hashed
  4. [vol 4] Logged-in users and cloud sessions           2 min (t= 17)  query user / who; note cloud tabs before they time out
  5. [vol 4] Running processes and open files             3 min (t= 20)  process list with command lines (e.g. tasklist /v, ps aux)
  6. [vol 5] Temporary files and swap/pagefile           10 min (t= 30)  collected inside the disk image; note pagefile/swap
  7. [vol 6] Disk image (with write blocker)             90 min (t=120)  imager + hardware write blocker, MD5+SHA-256  <- over time budget: hand to lab / preservation request
  8. [vol 7] Remote lo

## `session-03_law_matrix.py`

In [3]:
"""Session 3 - Which laws apply? A law-applicability matrix for fictional incidents.

Real incidents rarely involve one law. A breach at a Karachi company that serves
European customers can touch Pakistan's cybercrime law, its evidence law, the EU GDPR
and an industry standard all at once. This script describes each incident as a set of
facts ("tags"), keeps a small rule base of laws and the facts that trigger them, and
prints a matrix plus the reasons for every tick.

Teaching simplification only: the rule base is deliberately short and is NOT legal
advice. Laws are amended (Pakistan's PECA was amended in 2025, data-protection bills
change), so always check the current text and ask a lawyer.
Run:  python session-03_law_matrix.py
"""

# law code: (full name, kind, trigger tags (any one), what it gives the investigator)
# A cybercrime law only matters when a crime is suspected, so those laws also REQUIRE
# the tag "crime" (see REQUIRES). A purely accidental loss is a data-protection matter.
LAWS = {
    "PECA": ("Prevention of Electronic Crimes Act 2016 (Pakistan)", "cybercrime",
             {"offender_or_victim_pk", "system_in_pk"},
             "defines the offences (sections listed per incident below)"),
    "QSO+ETO": ("Qanun-e-Shahadat Order 1984 + Electronic Transactions Ordinance 2002 (Pakistan)",
                "evidence", {"court_in_pk"},
                "electronic documents and records are admissible; authenticity must be shown"),
    "GDPR": ("EU General Data Protection Regulation", "data protection",
             {"eu_personal_data"},
             "72-hour breach notice to the authority, data-subject rights, fines"),
    "CMA": ("Computer Misuse Act 1990 (UK)", "cybercrime", {"system_in_uk"},
            "unauthorised access and impairment offences"),
    "CFAA": ("Computer Fraud and Abuse Act (USA)", "cybercrime", {"system_in_us"},
             "unauthorised access to protected computers"),
    "HIPAA": ("Health Insurance Portability and Accountability Act (USA)", "sector privacy",
              {"us_health_data"}, "security and breach-notice duties for health data"),
    "PCI DSS": ("Payment Card Industry Data Security Standard", "contract standard",
                {"card_data"}, "not a law, but card brands and banks enforce it by contract"),
    "Budapest": ("Council of Europe Convention on Cybercrime (Budapest)", "treaty",
                 {"cross_border"},
                 "framework for cross-border evidence requests among parties; "
                 "check whether each country involved has joined"),
}

REQUIRES = {"PECA": {"crime"}, "CMA": {"crime"}, "CFAA": {"crime"}}

# Likely PECA sections per incident (teaching examples; a prosecutor decides the charge)
PECA_SECTIONS = {
    "A": "s3 unauthorised access, s4 unauthorised copying of data",
    "C": "s14 electronic fraud, s16 unauthorised use of identity information",
    "D": "s3 unauthorised access, s5 interference with data (the Pakistani firm's VPN was abused)",
    "E": "s16 identity misuse, s20 offences against dignity, s24 cyberstalking",
}

INCIDENTS = [
    ("A", "Ex-employee logs into a Karachi firm's ERP after leaving and copies the client list",
     {"crime", "offender_or_victim_pk", "system_in_pk", "court_in_pk"}),
    ("B", "Karachi call centre handling EU retail customers loses a laptop with their order history",
     {"system_in_pk", "eu_personal_data"}),
    ("C", "Phishing kit hosted in the UK steals card numbers from a Lahore online shop's customers",
     {"crime", "offender_or_victim_pk", "system_in_uk", "card_data", "cross_border", "court_in_pk"}),
    ("D", "Pakistani software house's US hospital client is hit by ransomware via a shared VPN",
     {"crime", "system_in_us", "us_health_data", "offender_or_victim_pk", "cross_border"}),
    ("E", "Student harasses a classmate with fake social-media profiles; case filed in Karachi",
     {"crime", "offender_or_victim_pk", "court_in_pk"}),
]


def applies(incident_tags, law_tags, code=None):
    if code and not REQUIRES.get(code, set()) <= incident_tags:
        return []
    return sorted(incident_tags & law_tags)


codes = list(LAWS)
print("Law applicability matrix (X = likely applies)\n")
print("Incident | " + " | ".join(f"{c:^8}" for c in codes))
print("-" * (11 + 11 * len(codes)))
for iid, _, tags in INCIDENTS:
    row = ["X" if applies(tags, LAWS[c][2], c) else "." for c in codes]
    print(f"   {iid}     | " + " | ".join(f"{r:^8}" for r in row))

print("\nWhy each tick:")
for iid, text, tags in INCIDENTS:
    print(f"\n{iid}. {text}")
    for c in codes:
        hit = applies(tags, LAWS[c][2], c)
        if hit:
            name, kind, _, gives = LAWS[c]
            if c == "PECA":
                gives = PECA_SECTIONS.get(iid, gives)
            print(f"   - {c} [{kind}] because {', '.join(hit)}: {gives}")

counts = {c: sum(1 for _, _, t in INCIDENTS if applies(t, LAWS[c][2], c)) for c in codes}
print("\nHow often each law was triggered:", ", ".join(f"{c}={n}" for c, n in counts.items()))
print("Lesson: one incident usually needs a cybercrime law (what is the offence?), an evidence")
print("law (will the court accept it?) and often a data-protection law (who must be told?).")

Law applicability matrix (X = likely applies)

Incident |   PECA   | QSO+ETO  |   GDPR   |   CMA    |   CFAA   |  HIPAA   | PCI DSS  | Budapest
---------------------------------------------------------------------------------------------------
   A     |    X     |    X     |    .     |    .     |    .     |    .     |    .     |    .    
   B     |    .     |    .     |    X     |    .     |    .     |    .     |    .     |    .    
   C     |    X     |    X     |    .     |    X     |    .     |    .     |    X     |    X    
   D     |    X     |    .     |    .     |    .     |    X     |    X     |    .     |    X    
   E     |    X     |    X     |    .     |    .     |    .     |    .     |    .     |    .    

Why each tick:

A. Ex-employee logs into a Karachi firm's ERP after leaving and copies the client list
   - PECA [cybercrime] because offender_or_victim_pk, system_in_pk: s3 unauthorised access, s4 unauthorised copying of data
   - QSO+ETO [evidence] because court_in_pk